# 🚀 Training YOLOv8n (Nano) — Dataset: wisard_ir

Notebook ini melatih model **YOLOv8n** menggunakan dataset **wisard_ir** pada GPU **T4** di Google Colab.

### ✨ Fitur
- **Auto-resume**: Jika runtime terputus, training otomatis dilanjutkan dari checkpoint terakhir
- **Checkpoint setiap 20 epoch**: Meminimalkan kehilangan progres
- **Auto-fix data.yaml**: Path dataset otomatis disesuaikan untuk Colab
- **Fast I/O**: Dataset di-extract dari ZIP ke disk lokal Colab (100x lebih cepat dari copy folder)

## Persiapan
1. Pastikan Runtime diatur ke **GPU T4**: `Runtime > Change runtime type > T4 GPU`
2. Pastikan file berikut ada di Google Drive:
   ```
   Google Drive/drone-wisard/datasets/wisard_ir.zip
   ```
   ATAU folder dataset yang sudah di-extract:
   ```
   Google Drive/drone-wisard/datasets/wisard_ir/
   ```
3. Jalankan semua cell secara berurutan
4. **Jika runtime terputus**: Cukup jalankan ulang SEMUA cell — training akan otomatis dilanjutkan dari checkpoint terakhir

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Install Dependencies

In [ ]:
!pip install -q ultralytics

## 3️⃣ Cek GPU & Setup Environment

In [ ]:
import os
import shutil
import torch
import yaml
import time
import zipfile

# Cek GPU
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.version.cuda}")
print(f"GPU tersedia: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB" if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"cuDNN       : {torch.backends.cudnn.version()}")
else:
    raise RuntimeError("❌ GPU tidak tersedia! Pastikan Runtime diatur ke GPU T4.")

## 4️⃣ Copy Dataset ke Disk Lokal (ZIP → Extract)

Google Drive punya latency tinggi (~170ms per file). Dengan 15.000+ file, copy folder bisa butuh **1+ jam**.

Strategi tercepat:
1. Copy 1 file ZIP dari Drive ke lokal (~5 menit)
2. Extract di SSD lokal (~1 menit)

> ⚡ Hanya perlu dilakukan **sekali per runtime session**. Jika sudah ada di lokal, akan di-skip.

In [ ]:
# ==============================
#   KONFIGURASI — Ubah jika perlu
# ==============================
MODEL_VARIANT = 'yolov8n'
DATASET_NAME  = 'wisard_ir'
RUN_NAME      = f'{MODEL_VARIANT}_{DATASET_NAME}'

# Path di Google Drive
DRIVE_ROOT    = '/content/drive/MyDrive/drone-wisard'
DRIVE_DATASET = os.path.join(DRIVE_ROOT, 'datasets', DATASET_NAME)
DRIVE_ZIP_1   = os.path.join(DRIVE_ROOT, 'datasets', f'{DATASET_NAME}.zip')  # datasets/wisard_ir.zip
DRIVE_ZIP_2   = os.path.join(DRIVE_ROOT, f'{DATASET_NAME}.zip')               # drone-wisard/wisard_ir.zip

# Path lokal Colab
LOCAL_BASE    = '/content/datasets'
LOCAL_DATASET = os.path.join(LOCAL_BASE, DATASET_NAME)
LOCAL_ZIP     = f'/content/{DATASET_NAME}.zip'

# Output training tetap di Google Drive agar persisten
PROJECT_PATH  = os.path.join(DRIVE_ROOT, 'runs_optimized')
COLLECTED_DIR = os.path.join(PROJECT_PATH, 'collected_best_models')

# --- Cek apakah dataset sudah ada di lokal ---
if os.path.exists(os.path.join(LOCAL_DATASET, 'images', 'train')):
    n_local = len(os.listdir(os.path.join(LOCAL_DATASET, 'images', 'train')))
    print(f"✅ Dataset sudah ada di lokal ({n_local} file). Skip copy.")

else:
    os.makedirs(LOCAL_BASE, exist_ok=True)

    # Prioritas: ZIP (1 file copy = cepat!) > folder (banyak file = lambat)
    zip_source = None
    if os.path.exists(DRIVE_ZIP_1):
        zip_source = DRIVE_ZIP_1
    elif os.path.exists(DRIVE_ZIP_2):
        zip_source = DRIVE_ZIP_2

    if zip_source:
        zip_size_gb = os.path.getsize(zip_source) / (1024**3)
        print(f"📦 Ditemukan ZIP: {zip_source} ({zip_size_gb:.1f} GB)")

        # Step 1: Copy ZIP ke lokal
        print(f"⏳ Step 1/2: Copy ZIP ke disk lokal...")
        start = time.time()
        shutil.copy2(zip_source, LOCAL_ZIP)
        copy_time = time.time() - start
        print(f"   ✅ Selesai dalam {copy_time:.0f} detik")

        # Step 2: Extract di lokal (SSD = sangat cepat)
        print(f"⏳ Step 2/2: Extract ZIP di disk lokal...")
        start = time.time()
        with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
            zf.extractall(LOCAL_BASE)
        extract_time = time.time() - start
        print(f"   ✅ Selesai dalam {extract_time:.0f} detik")

        # Hapus ZIP lokal untuk hemat disk
        os.remove(LOCAL_ZIP)
        print(f"   🗑️ ZIP lokal dihapus untuk hemat disk")

        # Cek apakah hasil extract ada di subfolder
        if not os.path.exists(os.path.join(LOCAL_DATASET, 'images')):
            # Mungkin extract ke folder berbeda, cari folder yang berisi 'images'
            for item in os.listdir(LOCAL_BASE):
                check_path = os.path.join(LOCAL_BASE, item, 'images')
                if os.path.isdir(check_path):
                    actual_path = os.path.join(LOCAL_BASE, item)
                    if actual_path != LOCAL_DATASET:
                        os.rename(actual_path, LOCAL_DATASET)
                        print(f"   📁 Renamed {item} → {DATASET_NAME}")
                    break

    elif os.path.exists(DRIVE_DATASET):
        # Fallback: copy folder (lebih lambat tapi tetap jalan)
        print(f"⚠️ ZIP tidak ditemukan, copy folder dari Drive (ini akan lebih lambat)...")
        print(f"   Sumber : {DRIVE_DATASET}")
        print(f"   Tujuan : {LOCAL_DATASET}")
        start = time.time()
        shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
        elapsed = time.time() - start
        print(f"   ✅ Selesai dalam {elapsed:.0f} detik")

    else:
        raise FileNotFoundError(
            f"❌ Dataset tidak ditemukan!\n"
            f"   Cek salah satu lokasi berikut di Google Drive:\n"
            f"   1. {DRIVE_ZIP_1}\n"
            f"   2. {DRIVE_ZIP_2}\n"
            f"   3. {DRIVE_DATASET}/"
        )

# Gunakan path lokal
DATASET_PATH = LOCAL_DATASET
DATA_YAML    = os.path.join(DATASET_PATH, 'data.yaml')

# Pastikan data.yaml ada
assert os.path.exists(DATA_YAML), f"❌ data.yaml tidak ditemukan di {DATA_YAML}"

# Fix data.yaml dengan path absolut lokal
with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = DATASET_PATH
data_cfg['train'] = 'images/train'
data_cfg['val']   = 'images/val'
data_cfg['test']  = 'images/test'

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

# Validasi
train_img_dir = os.path.join(DATASET_PATH, 'images', 'train')
n_train = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'))])

print(f"\n{'='*50}")
print(f"✅ Dataset path   : {DATASET_PATH} (LOKAL SSD — cepat!)")
print(f"📷 Jumlah gambar train: {n_train}")
print(f"📁 Output training: {os.path.join(PROJECT_PATH, RUN_NAME)} (Google Drive)")
print(f"🏆 Best model     : {os.path.join(COLLECTED_DIR, f'best_{RUN_NAME}.pt')}")
print(f"{'='*50}")

if n_train == 0:
    raise FileNotFoundError(f"❌ Folder {train_img_dir} kosong!")

## 5️⃣ Training YOLOv8n (dengan Auto-Resume)

| Parameter | Nilai |
|-----------|-------|
| Model | YOLOv8n (Nano) |
| Epochs | 400 |
| Image Size | 640 |
| Batch Size | 16 (optimal untuk T4 + YOLOv8n) |
| Patience | 100 |
| Checkpoint | Setiap 20 epoch |

> 🔄 **Auto-Resume**: Jika `last.pt` ditemukan dari training sebelumnya, training akan otomatis dilanjutkan

In [ ]:
from ultralytics import YOLO

torch.cuda.empty_cache()

# --- Auto-Resume: Cek apakah ada checkpoint dari training sebelumnya ---
LAST_PT = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'last.pt')
RESUME_TRAINING = os.path.exists(LAST_PT)

if RESUME_TRAINING:
    print("="*50)
    print("🔄 RESUME MODE — Melanjutkan training dari checkpoint terakhir")
    print(f"   Checkpoint: {LAST_PT}")
    print("="*50)
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
else:
    print("="*50)
    print("🆕 FRESH START — Memulai training dari awal")
    print("="*50)
    model = YOLO(f'{MODEL_VARIANT}.pt')

    results = model.train(
        data=DATA_YAML,

        # --- Core Training ---
        epochs=400,
        imgsz=640,
        batch=16,
        patience=100,
        device=0,
        workers=2,
        val=True,
        amp=True,
        deterministic=True,

        # --- Output ---
        project=PROJECT_PATH,
        name=RUN_NAME,
        exist_ok=True,
        save=True,
        save_period=20,
        plots=True,

        # --- Learning Rate ---
        lr0=0.01,
        lrf=0.01,
        cos_lr=True,
        warmup_epochs=5.0,

        # --- Augmentasi Khusus Infrared (IR) ---
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.1,

        # Augmentasi struktural (relevan untuk POV drone)
        degrees=15.0,
        translate=0.2,
        scale=0.5,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
        erasing=0.4,
    )

print("\n" + "="*50)
print(f"✅ Training {MODEL_VARIANT.upper()} selesai!")
print("="*50)

## 6️⃣ Salin Best Model ke Folder Terpusat

In [ ]:
best_src = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'best.pt')
os.makedirs(COLLECTED_DIR, exist_ok=True)
best_dst = os.path.join(COLLECTED_DIR, f'best_{RUN_NAME}.pt')

if os.path.exists(best_src):
    shutil.copy2(best_src, best_dst)
    print(f"✅ best.pt berhasil disalin!")
    print(f"   Sumber : {best_src}")
    print(f"   Tujuan : {best_dst}")
else:
    print(f"⚠️ WARNING: best.pt tidak ditemukan di {best_src}")

print(f"\n📊 Hasil training : {os.path.join(PROJECT_PATH, RUN_NAME)}")
print(f"🏆 Model terbaik  : {best_dst}")

if os.path.exists(best_dst):
    size_mb = os.path.getsize(best_dst) / (1024 * 1024)
    print(f"📦 Ukuran model   : {size_mb:.1f} MB")

## 7️⃣ Tampilkan Hasil Training

In [ ]:
from IPython.display import Image, display

results_dir = os.path.join(PROJECT_PATH, RUN_NAME)

plots = ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
         'P_curve.png', 'R_curve.png', 'F1_curve.png', 'PR_curve.png']

for plot in plots:
    plot_path = os.path.join(results_dir, plot)
    if os.path.exists(plot_path):
        print(f"\n📈 {plot}:")
        display(Image(filename=plot_path, width=800))